<a href="https://colab.research.google.com/github/palakagarwal01/spamfilter/blob/main/Spam_Filter_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the file with correct encoding to avoid errors
df = pd.read_csv('spam.csv', encoding='latin-1')

# Drop empty columns that Kaggle automatically adds
df = df.dropna(how="any", axis=1)

# Rename the columns to clean names
df.columns = ['label', 'text']

print("✅ Kaggle Dataset Loaded Successfully!")
print(df.head())


✅ Kaggle Dataset Loaded Successfully!
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
from sklearn.model_selection import train_test_split

# Convert text labels to numbers
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})

X = df['text']
y = df['label_num']

# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Data split completed!")


✅ Data split completed!


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize vectorizer and remove common stop words (the, is, a, etc.)
vectorizer = TfidfVectorizer(stop_words='english')

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("✅ Text converted to math vectors!")


✅ Text converted to math vectors!


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Train the model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Test the model
predictions = model.predict(X_test_tfidf)

print(f"🎉 Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")
print(classification_report(y_test, predictions, target_names=['Ham', 'Spam']))


🎉 Model Accuracy: 96.68%

              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       965
        Spam       1.00      0.75      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115



In [ ]:
def predict_new_message(msg):
    transformed = vectorizer.transform([msg])
    prediction = model.predict(transformed)
    return "🚨 SPAM" if prediction == 1 else "✅ HAM (Safe)"

# Test it out
print(predict_new_message("URGENT! You have won a 1,000 cash prize! Claim now."))
print(predict_new_message("Hey, are we still studying together tonight?"))



🚨 SPAM
✅ HAM (Safe)


In [8]:
import poplib
import email
from email.parser import Parser
from email.header import decode_header

# Configuration fields
POP_SERVER = "pop.gmail.com"  #  Correct address
EMAIL_ADDRESS = "pattu.261006@gmail.com"  # 👈 Update with your email address
APP_PASSWORD = "efwu aayn oqpb flka"   # 👈 Paste your 16-character App Password

try:
    print("🔄 Initializing connection to secure Google POP3 servers...")
    # Initialize connection via secure SSL layer channel on port 995
    mailbox = poplib.POP3_SSL(POP_SERVER, 995)
    mailbox.user(EMAIL_ADDRESS)
    mailbox.pass_(APP_PASSWORD)

    total_messages = len(mailbox.list())
    print(f"📥 Connection established! Server database total items: {total_messages}")
    print("🤖 Processing the last 5 emails through the ML classification engine...\n")
    print("=" * 75)

    # Process the last 5 indices sequentially from terminal tail positions
    start_index = max(1, total_messages - 4)
    for index in range(start_index, total_messages + 1):
        # Retrieve raw packet segments from server memory stack
        response, lines, octets = mailbox.retr(index)
        raw_content = b"\n".join(lines)

        # Parse data streams back into a clean structure map
        parsed_msg = Parser().parsestr(raw_content.decode('utf-8', errors='ignore'))

        # Isolate and decode headers to avoid formatting errors
        raw_subject = parsed_msg.get('Subject', '(No Subject Line Provided)')
        decoded_segments = decode_header(raw_subject)
        subject_output = ""
        for snippet, character_set in decoded_segments:
            if isinstance(snippet, bytes):
                subject_output += snippet.decode(character_set if character_set else 'utf-8', errors='ignore')
            else:
                subject_output += str(snippet)

        # Send live headers into your predictive functions
        inference_label = predict_new_message(subject_output)

        print(f"📧 INCOMING HEADER: {subject_output.strip()}")
        print(f"🎯 AI FILTER VERDICT: {inference_label}")
        print("=" * 75)

    mailbox.quit()
    print("\n🎉 Verification lifecycle complete! Production data verified successfully.")

except Exception as connection_error:
    print(f"\n❌ POP3 Processing Halt: {connection_error}")
    print("\n💡 Troubleshooting Fix Checklist:")
    print("- Ensure you saved the changes on your POP settings page in Gmail.")
    print("- Verify that your app password does not contain typos.")


🔄 Initializing connection to secure Google POP3 servers...
📥 Connection established! Server database total items: 3
🤖 Processing the last 5 emails through the ML classification engine...

📧 INCOMING HEADER: Security alert
🎯 AI FILTER VERDICT: ✅ HAM (Safe)
📧 INCOMING HEADER: 2-Step Verification turned on
🎯 AI FILTER VERDICT: ✅ HAM (Safe)
📧 INCOMING HEADER: Security alert
🎯 AI FILTER VERDICT: ✅ HAM (Safe)

🎉 Verification lifecycle complete! Production data verified successfully.
